# Regularization & cross-validation

_Machine Learning_

**Penalize big weights to fight overfitting. Use folds to tune the penalty.**

Overfitting often shows up as huge, wild weights.

     Regularization adds a penalty for big weights to the cost.

---

This notebook is a practice scaffold for the **Regularization & cross-validation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# few samples, many features -> plain least squares overfits
X, y = make_regression(n_samples=60, n_features=40, noise=10.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

ols = LinearRegression().fit(Xtr, ytr)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(Xtr, ytr)

print("OLS    test R^2:", round(r2_score(yte, ols.predict(Xte)), 3))
print("Ridge  test R^2:", round(r2_score(yte, ridge.predict(Xte)), 3))
print("chosen penalty alpha:", round(ridge.alpha_, 3))
print("OLS    weight norm:", round(float(np.linalg.norm(ols.coef_)), 2))
print("Ridge  weight norm:", round(float(np.linalg.norm(ridge.coef_)), 2))

## Visualize the data & results

_On diabetes data, does shrinking the weights (ridge) beat plain least squares, and how much penalty is too much?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

db = load_diabetes()
X = StandardScaler().fit_transform(db.data)
y = db.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

ols = LinearRegression().fit(Xtr, ytr)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(Xtr, ytr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(["OLS", "ridge (CV)"],
        [r2_score(yte, ols.predict(Xte)), r2_score(yte, ridge.predict(Xte))],
        color=["#ff7b72", "#7ee787"])
ax1.set_ylabel("test R squared")
ax1.set_title("OLS vs ridge")

alphas = np.logspace(-2, 3, 12)
r2s = [r2_score(yte, Ridge(alpha=a).fit(Xtr, ytr).predict(Xte)) for a in alphas]
ax2.plot(alphas, r2s, color="#c89bff", marker="o", label="test R squared")
ax2.set_xscale("log")
ax2.set_xlabel("alpha (penalty strength)")
ax2.set_ylabel("test R squared")
ax2.set_title("R squared vs ridge penalty")
ax2.legend()
plt.show()